# Fine-tuning FlanT5 pour l'extraction de voyages

## C'est quoi le fine-tuning ?

Imagine que FlanT5 est un étudiant qui a fait des études générales.
Le **fine-tuning**, c'est lui donner des cours particuliers sur les voyages en train.

```
AVANT: "Je veux aller de Paris à Lyon" → "" (ne comprend pas)
APRÈS: "Je veux aller de Paris à Lyon" → "DEPART: Paris | ARRIVEE: Lyon"
```

## Étape 0 : Activer le GPU

**IMPORTANT** : Va dans **Runtime > Change runtime type > GPU**

In [ ]:
import torch
if torch.cuda.is_available():
    print(f"GPU OK : {torch.cuda.get_device_name(0)}")
else:
    print("PAS DE GPU ! Va dans Runtime > Change runtime type > GPU")

## Étape 1 : Installer les bibliothèques

In [ ]:
!pip install -q transformers datasets accelerate sentencepiece

## Étape 2 : Uploader ton dataset

Upload le fichier `train.csv` depuis `TravelOrderResolver/datasets/base/train.csv`

In [ ]:
from google.colab import files
print("Clique sur 'Choose Files' et sélectionne train.csv")
uploaded = files.upload()

## Étape 3 : Préparer les données

On transforme le CSV en paires (input, output) pour FlanT5.

In [ ]:
import pandas as pd

df = pd.read_csv("train.csv")
print(f"Lignes: {len(df)}, Colonnes: {list(df.columns)}")
df.head()

In [ ]:
def create_example(row):
    sentence = row.get("sentence", row.get("text", ""))
    dep = row.get("departure", "") or "aucun"
    dest = row.get("destination", "") or "aucun"
    via = row.get("intermediate", "") or "aucun"
    
    input_text = f"Extrais les villes: {sentence}"
    output_text = f"DEPART: {dep} | ARRIVEE: {dest} | VIA: {via}"
    return {"input": input_text, "output": output_text}

training_data = [create_example(row) for _, row in df.iterrows()]
print(f"Exemples: {len(training_data)}")
print(f"Input:  {training_data[0]['input']}")
print(f"Output: {training_data[0]['output']}")

## Étape 4 : Charger FlanT5

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

MODEL = "google/flan-t5-base"
print(f"Chargement de {MODEL}...")

tokenizer = T5Tokenizer.from_pretrained(MODEL)
model = T5ForConditionalGeneration.from_pretrained(MODEL)
model = model.to("cuda")

print(f"Modèle chargé ! Paramètres: {model.num_parameters():,}")

## Étape 5 : Test AVANT fine-tuning

In [ ]:
def generate(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=64)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("=== AVANT FINE-TUNING ===")
test = "Extrais les villes: Je veux aller de Paris à Lyon"
print(f"Input:  {test}")
print(f"Output: {generate(test)}")

## Étape 6 : Tokenizer le dataset

In [ ]:
from datasets import Dataset

dataset = Dataset.from_list(training_data)

def tokenize(examples):
    inputs = tokenizer(examples["input"], max_length=256, truncation=True, padding="max_length")
    labels = tokenizer(examples["output"], max_length=64, truncation=True, padding="max_length")
    inputs["labels"] = labels["input_ids"]
    return inputs

tokenized = dataset.map(tokenize, batched=True, remove_columns=["input", "output"])
print(f"Dataset prêt: {len(tokenized)} exemples")

## Étape 7 : Entraînement (~15-30 min)

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

args = TrainingArguments(
    output_dir="./flan-t5-travel",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    learning_rate=5e-5,
    warmup_steps=100,
    logging_steps=100,
    save_steps=500,
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model),
)

print("Entraînement... (15-30 min)")
trainer.train()
print("Terminé !")

## Étape 8 : Test APRÈS fine-tuning

In [ ]:
print("=== APRÈS FINE-TUNING ===")
tests = [
    "Extrais les villes: Je veux aller de Paris à Lyon",
    "Extrais les villes: De Marseille à Nice via Toulon",
    "Extrais les villes: direction lille depuis paris",
]
for t in tests:
    print(f"Input:  {t.replace('Extrais les villes: ', '')}")
    print(f"Output: {generate(t)}")
    print()

## Étape 9 : Sauvegarder et télécharger

In [ ]:
model.save_pretrained("./flan-t5-travel-final")
tokenizer.save_pretrained("./flan-t5-travel-final")

!zip -r flan-t5-travel.zip flan-t5-travel-final/

from google.colab import files
files.download("flan-t5-travel.zip")

print("Télécharge le zip, puis:")
print("1. Décompresse dans TravelOrderResolver/models/flan-t5-travel/")
print("2. Le modèle est prêt !")